<a href="https://colab.research.google.com/github/ArpitaRPatki/NLP-LAB/blob/main/NLP_7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

In [2]:
# Sample Data: (Sentence, Tags)
# Labels: DET=Determiner, NN=Noun, V=Verb, ADV=Adverb, ADJ=Adjective
training_data = [
    ("the dog ate the apple".split(), ["DET", "NN", "V", "DET", "NN"]),
    ("everybody read the book".split(), ["NN", "V", "DET", "NN"]),
    ("i love you loads".split(), ["PRN", "V", "PRN", "ADV"])
]

In [3]:
# Create mappings
word_to_ix = {word: i for i, (sent, tags) in enumerate(training_data) for word in sent}
tag_to_ix = {"DET": 0, "NN": 1, "V": 2, "ADV": 3, "PRN": 4}

def prepare_sequence(seq, to_ix):
    return torch.tensor([to_ix[w] for w in seq], dtype=torch.long)

In [4]:
class LSTMTagger(nn.Module):
    def __init__(self, embedding_dim, hidden_dim, vocab_size, tagset_size):
        super(LSTMTagger, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)
        self.hidden2tag = nn.Linear(hidden_dim, tagset_size)

    def forward(self, sentence):
        embeds = self.embedding(sentence.unsqueeze(0)) # Add batch dim
        lstm_out, _ = self.lstm(embeds)
        # Apply linear layer to each time step
        tag_space = self.hidden2tag(lstm_out.squeeze(0))
        return tag_space

In [5]:
# Initialize
EMBEDDING_DIM = 6
HIDDEN_DIM = 6
model = LSTMTagger(EMBEDDING_DIM, HIDDEN_DIM, len(word_to_ix), len(tag_to_ix))
loss_function = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.1)

# Training (Simple loop)
for epoch in range(100):
    for sentence, tags in training_data:
        model.zero_grad()
        inputs = prepare_sequence(sentence, word_to_ix)
        targets = prepare_sequence(tags, tag_to_ix)

        tag_scores = model(inputs)
        loss = loss_function(tag_scores, targets)
        loss.backward()
        optimizer.step()

In [6]:
test_sentences = [
    "the dog ate the apple".split(),
    "i love you".split(),
    "everybody read".split(),
    "the apple".split(),
    "everybody love the apple".split(),
    "the dog read".split(),
    "i love the dog".split(),
    "the book".split(),
    "everybody ate".split(),
    "i ate".split()
]

model.eval()
with torch.no_grad():
    for sentence in test_sentences:
        inputs = prepare_sequence(sentence, word_to_ix)
        tag_scores = model(inputs)
        predictions = torch.argmax(tag_scores, dim=1)

        # Map indices back to tags
        ix_to_tag = {v: k for k, v in tag_to_ix.items()}
        predicted_tags = [ix_to_tag[idx.item()] for idx in predictions]
        print(f"{' '.join(sentence)}: {predicted_tags}")

the dog ate the apple: ['NN', 'NN', 'NN', 'NN', 'NN']
i love you: ['PRN', 'PRN', 'PRN']
everybody read: ['NN', 'NN']
the apple: ['NN', 'NN']
everybody love the apple: ['NN', 'V', 'NN', 'NN']
the dog read: ['NN', 'NN', 'NN']
i love the dog: ['PRN', 'PRN', 'V', 'V']
the book: ['NN', 'NN']
everybody ate: ['NN', 'NN']
i ate: ['PRN', 'V']
